In [2]:
import pandas as pd

customers_df = pd.read_json("noahs-customers.jsonl", lines=True)
orders_df = pd.read_json("noahs-orders.jsonl", convert_dates=['ordered', 'shipped'], lines=True)
products_df = pd.read_json("noahs-products.jsonl", lines=True)

SKU IDs:

- `PET` pets
- `COL` collectible
- `DLI` deli
- `BKY` bakery
- `HOM` household goods

In [3]:
norm_orders_df = orders_df.explode('items', ignore_index=True)
norm_orders_df = pd.concat([norm_orders_df.drop(columns=['items']), pd.json_normalize(norm_orders_df['items'])], axis=1)

In [4]:
df = norm_orders_df.join(
    customers_df.set_index('customerid'),
    on='customerid',
).join(products_df.set_index('sku'), on='sku')


Find orders of the bargain hunter

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 426541 entries, 0 to 426540
Data columns (total 19 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   orderid         426541 non-null  int64         
 1   customerid      426541 non-null  int64         
 2   ordered         426541 non-null  datetime64[us]
 3   shipped         426541 non-null  datetime64[us]
 4   total           426541 non-null  float64       
 5   sku             426541 non-null  str           
 6   qty             426541 non-null  int64         
 7   unit_price      426541 non-null  float64       
 8   name            426541 non-null  str           
 9   address         426541 non-null  str           
 10  citystatezip    426541 non-null  str           
 11  birthdate       426541 non-null  str           
 12  phone           426541 non-null  str           
 13  timezone        426541 non-null  str           
 14  lat             426541 non-null  float64       

In [17]:
df['desc_nocolor'] = df['desc'].str.replace(r" \(.*\)", "", regex=True)
df['ordered_date'] = df['ordered'].dt.date

In [18]:
bh_df = df[(df['customerid'] == 4167) & df['sku'].str.contains('COL')]
bh_df

,orderid,customerid,ordered,shipped,total,sku,qty,unit_price,name,address,...,birthdate,phone,timezone,lat,long,desc,wholesale_cost,dims_cm,desc_nocolor,ordered_date
138519,70503,4167,2018-12-31 12:26:58,2018-12-31 12:26:58,2.06,COL7063,1,2.06,Sherri Long,2092 Seward Ave,...,1975-04-09,585-838-9161,America/New_York,40.818198,-73.86752,Noah's Poster (azure),4.11,"[17.8, 15.2, 1.9]",Noah's Poster,2018-12-31
247339,124821,4167,2020-06-28 11:36:06,2020-06-28 11:36:06,13.04,COL7955,2,6.52,Sherri Long,2092 Seward Ave,...,1975-04-09,585-838-9161,America/New_York,40.818198,-73.86752,Noah's Action Figure (green),12.96,"[8.8, 6.8, 1.5]",Noah's Action Figure,2020-06-28
339995,170967,4167,2021-10-07 14:58:09,2021-10-07 20:15:00,6.65,COL6858,1,5.66,Sherri Long,2092 Seward Ave,...,1975-04-09,585-838-9161,America/New_York,40.818198,-73.86752,Noah's Jersey (blue),10.60,"[16.0, 6.6, 4.0]",Noah's Jersey,2021-10-07
379378,190610,4167,2022-04-23 14:25:36,2022-04-23 18:45:00,3.05,COL7663,1,2.06,Sherri Long,2092 Seward Ave,...,1975-04-09,585-838-9161,America/New_York,40.818198,-73.86752,Noah's Poster (white),4.07,"[5.6, 4.8, 4.1]",Noah's Poster,2022-04-23


In [21]:
pairings_df = df[df['customerid'] != 4167].merge(bh_df, on=['ordered_date', 'desc_nocolor'], suffixes=('_mc', '_bh'))
pairings_df

,orderid_mc,customerid_mc,ordered_mc,shipped_mc,total_mc,sku_mc,qty_mc,unit_price_mc,name_mc,address_mc,...,address_bh,citystatezip_bh,birthdate_bh,phone_bh,timezone_bh,lat_bh,long_bh,desc_bh,wholesale_cost_bh,dims_cm_bh
0,70476,7698,2018-12-31 08:38:36,2018-12-31 11:45:00,7.85,COL0837,1,4.57,Erika Mcconnell,8579 21st Ave,...,2092 Seward Ave,"Bronx, NY 10473",1975-04-09,585-838-9161,America/New_York,40.818198,-73.86752,Noah's Poster (azure),4.11,"[17.8, 15.2, 1.9]"
1,70477,2610,2018-12-31 08:54:05,2018-12-31 15:30:00,11.82,COL5018,1,6.17,Karen Contreras,169 Allen St,...,2092 Seward Ave,"Bronx, NY 10473",1975-04-09,585-838-9161,America/New_York,40.818198,-73.86752,Noah's Poster (azure),4.11,"[17.8, 15.2, 1.9]"
2,70502,5783,2018-12-31 12:26:39,2018-12-31 12:26:39,4.57,COL2467,1,4.57,Carlos Myers,1486 Bath Ave,...,2092 Seward Ave,"Bronx, NY 10473",1975-04-09,585-838-9161,America/New_York,40.818198,-73.86752,Noah's Poster (azure),4.11,"[17.8, 15.2, 1.9]"
3,70507,7474,2018-12-31 13:02:52,2018-12-31 13:02:52,4.27,COL2467,1,4.27,Alex Evans,110B Warren St,...,2092 Seward Ave,"Bronx, NY 10473",1975-04-09,585-838-9161,America/New_York,40.818198,-73.86752,Noah's Poster (azure),4.11,"[17.8, 15.2, 1.9]"
4,124848,6045,2020-06-28 14:59:03,2020-06-28 16:45:00,56.15,COL3038,1,16.12,Sheila Lopez,592 Throop Ave,...,2092 Seward Ave,"Bronx, NY 10473",1975-04-09,585-838-9161,America/New_York,40.818198,-73.86752,Noah's Action Figure (green),12.96,"[8.8, 6.8, 1.5]"
5,190619,1162,2022-04-23 15:31:27,2022-04-23 15:31:27,11.86,COL7063,1,4.71,Jeffrey Johnson,385½ E 74th St,...,2092 Seward Ave,"Bronx, NY 10473",1975-04-09,585-838-9161,America/New_York,40.818198,-73.86752,Noah's Poster (white),4.07,"[5.6, 4.8, 4.1]"


In [28]:
pairings_df[pairings_df['sku_mc'] != pairings_df['sku_bh']][['desc_bh', 'desc_mc', 'customerid_mc', 'ordered_mc', 'ordered_bh']]

,desc_bh,desc_mc,customerid_mc,ordered_mc,ordered_bh
0,Noah's Poster (azure),Noah's Poster (mauve),7698,2018-12-31 08:38:36,2018-12-31 12:26:58
1,Noah's Poster (azure),Noah's Poster (puce),2610,2018-12-31 08:54:05,2018-12-31 12:26:58
2,Noah's Poster (azure),Noah's Poster (orange),5783,2018-12-31 12:26:39,2018-12-31 12:26:58
3,Noah's Poster (azure),Noah's Poster (orange),7474,2018-12-31 13:02:52,2018-12-31 12:26:58
4,Noah's Action Figure (green),Noah's Action Figure (purple),6045,2020-06-28 14:59:03,2020-06-28 11:36:06
5,Noah's Poster (white),Noah's Poster (azure),1162,2022-04-23 15:31:27,2022-04-23 14:25:36


In [29]:
df[df['customerid'] == 5783]

,orderid,customerid,ordered,shipped,total,sku,qty,unit_price,name,address,...,birthdate,phone,timezone,lat,long,desc,wholesale_cost,dims_cm,desc_nocolor,ordered_date
34591,18431,5783,2017-07-25 11:13:20,2017-07-25 11:13:20,3.77,PET9308,1,1.67,Carlos Myers,1486 Bath Ave,...,1986-04-27,838-335-7157,America/New_York,40.608261,-74.013753,"Vegan Adult Cat Food, Tuna & Tuna",1.40,"[17.0, 6.7, 4.0]","Vegan Adult Cat Food, Tuna & Tuna",2017-07-25
34592,18431,5783,2017-07-25 11:13:20,2017-07-25 11:13:20,3.77,PET4482,1,2.10,Carlos Myers,1486 Bath Ave,...,1986-04-27,838-335-7157,America/New_York,40.608261,-74.013753,"Gluten-free Cat Food, Chicken & Chicken",1.77,"[17.2, 14.1, 9.7]","Gluten-free Cat Food, Chicken & Chicken",2017-07-25
38481,20430,5783,2017-08-13 17:38:18,2017-08-13 17:38:18,1.38,PET3942,1,1.38,Carlos Myers,1486 Bath Ave,...,1986-04-27,838-335-7157,America/New_York,40.608261,-74.013753,Snake Food,1.14,"[18.0, 16.5, 1.9]",Snake Food,2017-08-13
87068,44846,5783,2018-04-17 18:31:15,2018-04-17 18:31:15,9.19,PET0759,1,1.07,Carlos Myers,1486 Bath Ave,...,1986-04-27,838-335-7157,America/New_York,40.608261,-74.013753,"Dry Senior Cat Food, Chicken & Chicken",0.97,"[19.0, 5.7, 4.7]","Dry Senior Cat Food, Chicken & Chicken",2018-04-17
87069,44846,5783,2018-04-17 18:31:15,2018-04-17 18:31:15,9.19,TOY5805,1,8.12,Carlos Myers,1486 Bath Ave,...,1986-04-27,838-335-7157,America/New_York,40.608261,-74.013753,Star Wars Hebrew Blocks,6.43,"[17.9, 11.2, 7.0]",Star Wars Hebrew Blocks,2018-04-17
128908,65739,5783,2018-11-12 22:48:04,2018-11-18 18:30:00,1.23,PET1976,1,1.23,Carlos Myers,1486 Bath Ave,...,1986-04-27,838-335-7157,America/New_York,40.608261,-74.013753,"Dry Adult Cat Food, Beef & Beef",0.95,"[13.7, 8.9, 4.0]","Dry Adult Cat Food, Beef & Beef",2018-11-12
138518,70502,5783,2018-12-31 12:26:39,2018-12-31 12:26:39,4.57,COL2467,1,4.57,Carlos Myers,1486 Bath Ave,...,1986-04-27,838-335-7157,America/New_York,40.608261,-74.013753,Noah's Poster (orange),3.58,"[16.3, 6.1, 4.7]",Noah's Poster,2018-12-31
141429,71940,5783,2019-01-14 14:22:24,2019-01-14 18:00:00,4.56,PET0769,2,2.28,Carlos Myers,1486 Bath Ave,...,1986-04-27,838-335-7157,America/New_York,40.608261,-74.013753,"Gluten-free Senior Cat Food, Turkey & Turkey",1.50,"[17.8, 15.7, 5.5]","Gluten-free Senior Cat Food, Turkey & Turkey",2019-01-14
229138,115807,5783,2020-03-27 17:44:00,2020-03-29 16:15:00,11.68,COL7454,1,10.41,Carlos Myers,1486 Bath Ave,...,1986-04-27,838-335-7157,America/New_York,40.608261,-74.013753,Noah's Jersey (mauve),8.19,"[19.1, 18.2, 3.5]",Noah's Jersey,2020-03-27
229139,115807,5783,2020-03-27 17:44:00,2020-03-29 16:15:00,11.68,PET9602,1,1.27,Carlos Myers,1486 Bath Ave,...,1986-04-27,838-335-7157,America/New_York,40.608261,-74.013753,"Dry Senior Cat Food, Salmon & Salmon",1.06,"[16.2, 13.5, 10.8]","Dry Senior Cat Food, Salmon & Salmon",2020-03-27
